# Análisis de logs distribuidos

Objetivo: Detectar el momento crítico del sistema y explicar qué servicio, endpoint y mensajes fueron los más afectados

# Imports
### "Pandas" --> Utilizado para trabajar con el file .csv 
-Cargar el archivo.  
-Convertir columnas.  
-Agrupar por ventanas de tiempo.  
-Conteo de eventos.  
-Cálculo de operaciones (%)  
-Creación de tablas de resumen
### "Matplotlib" --> Utilizado para la creación de los gráficos
-Se aplica para visualizar tendencias temporales.


In [19]:
import pandas as pd
import matplotlib.pyplot as plt

# Carga y validacion de datos
Se carga el file .CSV como un DataFrame y se validan los datos antes de empezar a trabajar con ellos.  
Un DataFrame es una tabla en memoria, donde cada fila representa un log y cada columna es una categoría de datos relevantes para el análisis, tales como:   
-El horario en el que se produjo el evento.  
-El nombre del servicio afectado.  
-La severidad del incidente.  
-El endpoint afectado.  
-El status code.  
-La latencia.  
-etc

In [24]:
# Bloque de carga y validación de datos

df = pd.read_csv("server_logs.csv") # --> Carga del file "server_logs.csv" en un DataFrame

# Estas son las 10 columnas obligatorias que solicita el challenge
required_columns = [
    "timestamp_event",
    "received_at",
    "service_name",
    "severity",
    "message",
    "method",
    "endpoint",
    "status_code",
    "latency_ms",
    "trace_id",
]

# Variable de control de cantidad (columnas mínimas)
missing_columns = sorted(set(required_columns) - set(df.columns)) # Se aplica sets para poder hacer operaciones.
if missing_columns:
    raise ValueError(f"Faltan columnas obligatorias: {missing_columns}") # --> Se detiene con un error porque no están las columnas mínimas.

# Conversión de datatypes
df["timestamp_event"] = pd.to_datetime(df["timestamp_event"], utc=True) #datetime
df["received_at"] = pd.to_datetime(df["received_at"], utc=True) #datetime
df["status_code"] = pd.to_numeric(df["status_code"], errors="coerce") #numeric
df["latency_ms"] = pd.to_numeric(df["latency_ms"], errors="coerce") #numeric
df = df.sort_values("timestamp_event").reset_index(drop=True) # Se ordena de la fecha + antigua a la reciente. Se resetean los índices y se genera uno nuevo según el sorting.
# errors = coerce --> Suponiendo que se completa con algo no esperado como un "ERROR", "texto:corrupto" o "espacio en blanco" pasa a un valor vacío NaN (Not a Number)

# Prints para indicar la cantidad de logs del dataset cargado y el rango temporal a analizar
print(f"Dataset cargado: {len(df):,} logs")
print(f"Rango temporal: {df['timestamp_event'].min()} -> {df['timestamp_event'].max()}")

Dataset cargado: 5,795 logs
Rango temporal: 2026-01-10 00:02:39.029160+00:00 -> 2026-01-12 23:59:23.187914+00:00


# Definiciones operativas obligatorias y definición de un "bad_event"

In [32]:
# Bloque de definición de flujo operativo

# Agrupación por bloques definidos temporalmente cada 5 minutos (Si algo ocurre a las 19:24:48 se le asigna la ventana bin de 19:20:00)
df["window_start"] = df["timestamp_event"].dt.floor("5min") # --> Cada log se asigna a una ventana de 5 minutos usando `timestamp_event`.
# Identificación de errores 5XX
df["is_5xx"] = df["status_code"] >= 500 # --> Log marcado como >= 500, lo que supone un error del lado del server.
# Definición y condiciones del bad event
df["is_bad_event"] = df["severity"].isin(["ERROR", "CRITICAL"]) | df["is_5xx"] # --> Se considera "bad_event" si su severidad es ERROR o CRITICAL o su status code >= 500.

# Se crea un df de "bad_event"
bad_df = df[df["is_bad_event"]].copy() # --> Esto va a usarse para el cálculo de mensajes de error y diagnóstico del incidente.

# Impresión de bad_events y % sobre el total de logs
print(f"Bad events totales: {df['is_bad_event'].sum():,}")
print(f"Bad rate global: {df['is_bad_event'].mean():.2%}")
print("-" * 50)

# Agrupamos por ventana para calcular el total de eventos y los eventos malos
# (En Python, sumar un booleano True/False cuenta los True automáticamente)
window_summary = df.groupby("window_start").agg(
    total_events=("is_bad_event", "count"),
    bad_events=("is_bad_event", "sum")
).reset_index()

# 3. BAD RATE (Por ventana)
# Calculamos la tasa directamente dividiendo las columnas de Pandas
window_summary["bad_rate"] = window_summary["bad_events"] / window_summary["total_events"]

# 4. MOMENTO CRÍTICO
# Aplicamos el filtro obligatorio: ventanas con 20 o más eventos totales
filtered_windows = window_summary[window_summary["total_events"] >= 20]

if not filtered_windows.empty:
    # Encontramos la fila con el bad_rate más alto usando idxmax de Pandas
    critical_row_idx = filtered_windows["bad_rate"].idxmax()
    critical_row = filtered_windows.loc[critical_row_idx]
    
    momento_critico_window = critical_row["window_start"]
    
    print(f"Momento Crítico Detectado:")
    print(f"Ventana: {momento_critico_window}")
    print(f"Eventos totales en la ventana: {critical_row['total_events']}")
    print(f"Bad Rate de la ventana: {critical_row['bad_rate']:.2%}")
else:
    momento_critico_window = None
    print("⚠️ No se encontraron ventanas con >= 20 eventos. No se puede definir un Momento Crítico.")

print("-" * 50)

# 5. BASELINE
# Dividimos el dataset original usando filtros booleanos de Pandas
if momento_critico_window is not None:
    df_momento_critico = df[df["window_start"] == momento_critico_window].copy()
    df_baseline = df[df["window_start"] != momento_critico_window].copy()
    
    print(f"Flujo Operativo Segmentado con Éxito:")
    print(f"Registros en Momento Crítico: {len(df_momento_critico):,}")
    print(f"Registros en Baseline (Resto del dataset): {len(df_baseline):,}")
else:
    df_baseline = df.copy()
    print("ℹ️ Al no haber momento crítico, el Baseline incluye el 100% de los datos.")


Bad events totales: 895
Bad rate global: 15.44%
--------------------------------------------------
Momento Crítico Detectado:
Ventana: 2026-01-10 11:10:00+00:00
Eventos totales en la ventana: 189
Bad Rate de la ventana: 58.20%
--------------------------------------------------
Flujo Operativo Segmentado con Éxito:
Registros en Momento Crítico: 189
Registros en Baseline (Resto del dataset): 5,606


# 1- Exploración inicial
Responde con tablas a las preguntas:  
1) ¿Cuántos logs hay en total?
2) ¿Qué severidad aparece más?
3) ¿Qué servicio genera más logs? (Y el que genera menos)
4) ¿Cuál es el mensaje más repetido?
5) ¿Cuál es el mensaje "malo" más repetido (bad events)?

In [27]:
message_counts = df["message"].value_counts()
bad_message_counts = bad_df["message"].value_counts()
service_counts = df["service_name"].value_counts()

exploracion_inicial = pd.DataFrame(
    [
        {"pregunta": "Logs totales", "respuesta": len(df)},
        {"pregunta": "Severidad mas frecuente", "respuesta": df["severity"].value_counts().idxmax()},
        {"pregunta": "Servicio con mas logs", "respuesta": service_counts.idxmax()},
        {"pregunta": "Servicio con menos logs", "respuesta": service_counts.idxmin()},
        {"pregunta": "Mensaje mas repetido", "respuesta": message_counts.idxmax()},
        {
            "pregunta": "Mensaje malo mas repetido",
            "respuesta": bad_message_counts.idxmax() if not bad_message_counts.empty else "Sin bad events",
        },
    ]
)

exploracion_inicial

,pregunta,respuesta
0,Logs totales,5795
1,Severidad mas frecuente,INFO
2,Servicio con mas logs,api-gateway
3,Servicio con menos logs,notification-service
4,Mensaje mas repetido,Health check OK
5,Mensaje malo mas repetido,Order creation failed - inventory lock timeout


# 2- Deteccion del momento critico

In [33]:
window_summary = (
    df.groupby("window_start")
    .agg(
        total_events=("trace_id", "size"),
        bad_events=("is_bad_event", "sum"),
        avg_latency_ms=("latency_ms", "mean"),
        events_5xx=("is_5xx", "sum"),
    )
    .reset_index()
)
window_summary["bad_rate"] = window_summary["bad_events"] / window_summary["total_events"]
window_summary["pct_5xx"] = window_summary["events_5xx"] / window_summary["total_events"]

eligible_windows = window_summary[window_summary["total_events"] >= 20].copy()
if eligible_windows.empty:
    raise ValueError("No hay ventanas de 5 minutos con total_events >= 20")

top_5_windows = (
    eligible_windows.sort_values(
        ["bad_rate", "bad_events", "total_events"],
        ascending=[False, False, False],
    )
    .head(5)
    [["window_start", "total_events", "bad_events", "bad_rate"]]
    .reset_index(drop=True)
)

critical_window_start = top_5_windows.loc[0, "window_start"]
critical_window_end = critical_window_start + pd.Timedelta(minutes=5)

print(f"Momento critico seleccionado: {critical_window_start} -> {critical_window_end}")
top_5_windows

Momento critico seleccionado: 2026-01-10 11:10:00+00:00 -> 2026-01-10 11:15:00+00:00


,window_start,total_events,bad_events,bad_rate
0,2026-01-10 11:10:00+00:00,189,110,0.582011
1,2026-01-10 11:15:00+00:00,228,129,0.565789
2,2026-01-10 11:20:00+00:00,111,59,0.531532
3,2026-01-11 14:35:00+00:00,255,117,0.458824
4,2026-01-11 14:30:00+00:00,156,68,0.435897
